In [1]:
%load_ext autoreload
%autoreload 2

import torch
from attention import TinyGPT
from data_gathering import gather_data, format_data
from training import train, generate_text, uniform_random_loss
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__, "| CUDA:", torch.cuda.is_available())

2.11.0+cu128 | CUDA: True


In [2]:
# Fetches Wikipedia pages, Gutenberg books, and Shakespeare corpus.
# Pass custom lists to override the defaults (see attention.py).
# text = gather_data()

with open("dog_corpus.txt", "r", encoding="utf-8") as f:
    text = f.read()


In [3]:
seq_len = 64
input_ids, target_ids, vocab, stoi, itos = format_data(text, seq_len=seq_len)

Vocab size : 47,564
Sequences  : 559,540  (seq_len=64)


In [4]:
input_ids  = input_ids.to(device)
target_ids = target_ids.to(device)

split_idx = int(0.9 * input_ids.size(0))
train_input_ids,  val_input_ids  = input_ids[:split_idx],  input_ids[split_idx:]
train_target_ids, val_target_ids = target_ids[:split_idx], target_ids[split_idx:]

print("Train:", train_input_ids.shape, "| Val:", val_input_ids.shape)

Train: torch.Size([503586, 64]) | Val: torch.Size([55954, 64])


In [12]:
model = TinyGPT(
    vocab_size=len(vocab),
    d_model=64,
    hidden_dim=256,
    num_layers=4,
    max_len=seq_len,
    dropout=0.1
).to(device)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 3,244,160


In [6]:
uniform = uniform_random_loss(len(vocab))

print(f"Vocab size:            {len(vocab):,}")
print(f"Uniform random loss:   {uniform:.4f}   (knows nothing)")


Vocab size:            47,564
Uniform random loss:   10.7698   (knows nothing)


In [7]:
train_losses, val_losses, train_accs, val_accs = train(
    model,
    train_input_ids, train_target_ids,
    val_input_ids,   val_target_ids,
    num_epochs=15,
    batch_size=32,
    device=device,
)

Epoch 01/15 | Train Loss: 7.2427  Acc: 0.0750 | Val Loss: 6.4198  Acc: 0.1409 | LR: 2.00e-04
Epoch 02/15 | Train Loss: 5.7656  Acc: 0.1576 | Val Loss: 6.0910  Acc: 0.1518 | LR: 2.99e-04
Epoch 03/15 | Train Loss: 5.3958  Acc: 0.1867 | Val Loss: 6.1461  Acc: 0.1533 | LR: 2.91e-04
Epoch 04/15 | Train Loss: 5.3121  Acc: 0.1936 | Val Loss: 6.0612  Acc: 0.1546 | LR: 2.75e-04
Epoch 05/15 | Train Loss: 5.3125  Acc: 0.1925 | Val Loss: 6.1351  Acc: 0.1586 | LR: 2.53e-04
Epoch 06/15 | Train Loss: 5.2938  Acc: 0.1939 | Val Loss: 6.1048  Acc: 0.1535 | LR: 2.25e-04
Epoch 07/15 | Train Loss: 5.2908  Acc: 0.1939 | Val Loss: 6.0842  Acc: 0.1573 | LR: 1.93e-04
Epoch 08/15 | Train Loss: 5.2820  Acc: 0.1954 | Val Loss: 6.5432  Acc: 0.1549 | LR: 1.59e-04
Epoch 09/15 | Train Loss: 5.3000  Acc: 0.1938 | Val Loss: 6.1218  Acc: 0.1567 | LR: 1.24e-04
Epoch 10/15 | Train Loss: 5.3113  Acc: 0.1919 | Val Loss: 6.2442  Acc: 0.1555 | LR: 9.06e-05
Epoch 11/15 | Train Loss: 5.3139  Acc: 0.1926 | Val Loss: 6.3694  Acc:

KeyboardInterrupt: 

In [ ]:
prompt = "Maki is small and cute english toy terrier"

In [ ]:
print(generate_text(model, prompt, stoi, itos, seq_len, device, max_new_tokens=25))

is small and english toy terrier beagle , a mastiff type of a mastiff .” the been—yes , by jove afford a curly-haired spaniel . this was risen and paced along
